# Interactive browser login (Authorization Code + PKCE)

The SDK equivalent of the CLI's `b2c auth login <clientId>`. It opens your
browser, runs the Authorization Code + PKCE flow against Account Manager, and
**persists the session** to the shared `auth-sessions.json` store — the same
store the `b2c` CLI uses, so the session minted here is interchangeable with a
`b2c auth login` session (and with the `04-cli-session` notebook).

Needs only `clientId` in `../dw.json` (no client secret). The client's
registered redirect URI must include `http://localhost:8080` (the default port).

> ⚠️ **This opens a real browser window and waits for you to approve the login.**

In [ ]:
from pathlib import Path

from b2c_tooling_sdk import CreateB2CInstanceOptions, ResolveConfigOptions, list_code_versions, resolve_config
from b2c_tooling_sdk.auth import PkceOAuthConfig, create_user_auth_strategy, find_auth_session

import json

DW_JSON = Path("../dw.json").resolve()
raw = json.loads(DW_JSON.read_text())
client_id = raw["clientId"]
account_manager_host = raw.get("accountManagerHost") or "account.demandware.com"

# The browser-based "user" strategy: PKCE, with an implicit fallback for clients
# not registered for PKCE — the same primitive `b2c auth login` uses.
strategy = create_user_auth_strategy(
    PkceOAuthConfig(client_id=client_id, account_manager_host=account_manager_host)
)

In [ ]:
# Opens the browser and blocks until you approve, then saves the session.
token = await strategy.get_token_response()
print(f"Login succeeded. Token expires at {token.expires:%Y-%m-%d %H:%M:%S %Z}.")

# The session now lives in the shared store — reusable by this SDK or the b2c CLI.
if find_auth_session(client_id) is not None:
    print("Session persisted (reusable by `b2c` and the 04-cli-session notebook).")

## Verify the fresh session with a real OCAPI call

Reuse the strategy we just built to list the instance's code versions.

In [ ]:
config = await resolve_config(options=ResolveConfigOptions(config_path=str(DW_JSON)))
instance = config.create_b2c_instance(CreateB2CInstanceOptions(oauth_strategy=strategy))

versions = await list_code_versions(instance)
print(f"{len(versions)} code version(s) on {raw['hostname']}:")
for version in versions:
    print("  ", version.id, "(active)" if version.active else "")